# Solutions · 02 · Rheometers and their pitfalls

Worked solutions to the exercises of [notebook 02](../notebooks/02_rheometers_and_pitfalls.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engrheo is missing): install it from GitHub
    import engrheo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engrheo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engrheo import datasets, fitting, geometry, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
melt = pd.read_csv(datasets.path("polymer_melt_capillary.csv"), comment="#")
melt["L_over_R"] = melt.die_length_mm / melt.die_radius_mm
Rd = melt.die_radius_mm.iloc[0] * 1e-3

## Exercise 1

Fit a Carreau model to the melt data three ways - raw single-die data, Bagley-corrected data at the apparent rate, and fully corrected data - and compare $\eta_0$ and $n$ with the true values (3000 Pa·s, 0.3).

In [3]:
tau_w, app = [], []
for Q, grp in melt.groupby("flow_rate_m3_s"):
    tau_w.append(geometry.bagley(grp.L_over_R, grp.pressure_drop_Pa)["tau_w"]); app.append(4 * Q / (np.pi * Rd**3))
tau_w, app = np.array(tau_w), np.array(app)
_, rate_w, _ = geometry.capillary(2 * tau_w, app * np.pi * Rd**3 / 4, R=Rd, L=Rd)
single = melt[melt.L_over_R == 10]
raw_rate = (4 * single.flow_rate_m3_s / (np.pi * Rd**3)).to_numpy()
raw_tau = (single.pressure_drop_Pa * Rd / (2 * 10 * Rd)).to_numpy()
grid = np.logspace(np.log10(rate_w.min()), np.log10(rate_w.max()), 50)
true = models.viscosity("carreau", grid, 3000.0, 0.0, 0.3, 0.3)
for label, r_, t_ in (("raw single die", raw_rate, raw_tau), ("Bagley only", app, tau_w), ("Bagley + Rabinowitsch", rate_w, tau_w)):
    f = fitting.fit_flow_curve(r_, t_, "carreau", p0=[3000, 0.0, 0.3, 0.3])
    err = f.predict_viscosity(grid) / true - 1
    print(f"{label:<22}: eta0 = {f.params['eta0']:5.0f} Pa s, n = {f.params['n']:.3f}, lambda = {f.params['lam']:.3f} s | "
          f"largest viscosity error within the measured range {np.max(np.abs(err)):6.1%}")
print("true: eta0 = 3000 Pa s, n = 0.30, lambda = 0.30 s")

raw single die        : eta0 =  3371 Pa s, n = 0.335, lambda = 0.190 s | largest viscosity error within the measured range 104.9%
Bagley only           : eta0 =  2447 Pa s, n = 0.295, lambda = 0.179 s | largest viscosity error within the measured range  15.6%
Bagley + Rabinowitsch : eta0 =  1888 Pa s, n = 0.282, lambda = 0.143 s | largest viscosity error within the measured range   2.1%
true: eta0 = 3000 Pa s, n = 0.30, lambda = 0.30 s


Judged by $\eta_0$, the raw data seem best - but that is a coincidence. These data start at 10 1/s, above the onset
of shear thinning ($1/\lambda$ ≈ 3 1/s), so $\eta_0$ and $\lambda$ are extrapolations that none of the data sets can determine.
The meaningful test is the viscosity **within the measured range**: there the raw single-die curve is wrong by
up to about 100 % (it counts the entrance pressure loss as die friction), the Bagley-corrected one by about
15 %, and the fully corrected one by about 2 %. Compare models where the data are; capillary rheometers are
poorly suited to measuring zero-shear viscosities.

## Exercise 2

Taylor vortices appear in a Couette cell when $Ta = \Omega^2 R_i (R_o - R_i)^3 / \nu^2$ exceeds about 1700. At what rotation speed does this happen for water in a cell with $R_i$ = 12.5 mm, $R_o$ = 13.6 mm, and what shear rate does that correspond to?

In [4]:
Ri, Ro, nu = 0.0125, 0.0136, 1.0e-6
d = Ro - Ri
Omega = np.sqrt(1700 * nu**2 / (Ri * d**3))
print(f"Taylor vortices above Omega = {Omega:.1f} rad/s ({Omega*60/(2*np.pi):.0f} rpm), i.e. a shear rate of about {Omega*Ri/d:.0f} 1/s")

Taylor vortices above Omega = 10.1 rad/s (97 rpm), i.e. a shear rate of about 115 1/s


For water, secondary flows set in at a shear rate of only about 100 1/s in this cell. Above it, the extra
dissipation of the vortices raises the torque and the "viscosity" appears to increase - an apparent shear
thickening that is purely an artefact. Low-viscosity fluids at high rates need narrower gaps or other
geometries.

## Exercise 3

**Implement it yourself:** do the Mooney analysis for one stress level of the toothpaste data by hand (straight-line fit of apparent rate against 1/gap) and compare with `geometry.mooney`.

In [5]:
paste = pd.read_csv(datasets.path("toothpaste_plates_slip.csv"), comment="#")
grp = paste[paste.shear_stress_Pa == paste.shear_stress_Pa.unique()[2]]
slope, intercept = np.polyfit(1 / (grp.gap_mm * 1e-3), grp.apparent_shear_rate_1_s, 1)
lib = geometry.mooney(grp.gap_mm * 1e-3, grp.apparent_shear_rate_1_s, kind="plates")
print(f"stress {grp.shear_stress_Pa.iloc[0]} Pa: by hand true rate {intercept:.4f} 1/s, slip {slope/2*1e3:.4f} mm/s; "
      f"library {lib['true_rate']:.4f} 1/s, {lib['slip_velocity']*1e3:.4f} mm/s")

stress 100.1 Pa: by hand true rate 0.0108 1/s, slip 0.3926 mm/s; library 0.0108 1/s, 0.3926 mm/s


For parallel plates the apparent rim rate is the true rate plus $2u_s/h$; a straight line against $1/h$ has
intercept equal to the true rate and slope $2u_s$ - the same calculation as the library's.